In [ ]:
import scanpy as sc
import pandas as pd
import infercnvpy as cnv
import numpy as np
from sklearn.mixture import GaussianMixture
#import rapids_singlecell as rsc

In [ ]:
adata = sc.read_h5ad('../adata_qc.h5ad')
adata_anno = sc.read_h5ad('../adata_anno_cell_subtype_re.h5ad')

In [ ]:
adata_epi_umap = sc.read_h5ad('../leiden_detailed/adata_epi.h5ad')
adata_epi_umap

In [ ]:
cols = ["leiden_coarse", "cell_type", "cell_type_fine",'cell_subtype']
# 先用 adata_anno 过滤 adata
adata = adata[adata_anno.obs_names].copy()

# 按 obs_names 对齐复制三列
adata.obs[cols] = adata_anno.obs.loc[adata.obs_names, cols]

# 去除这三列里有 NA 的细胞
adata = adata[adata.obs[cols].notna().all(axis=1)].copy()
adata

In [ ]:
adata_epi = adata[adata.obs['cell_type']=='Epi',:].copy()
adata_epi

In [ ]:
(adata_epi.obs_names == adata_epi_umap.obs_names).all()

In [ ]:
adata_epi.obsm["X_umap"] = adata_epi_umap[adata_epi.obs_names].obsm["X_umap"].copy()

In [ ]:
# ========= 2. 读取 GTF 注释 =========
gtf_file = "gencode.v49.basic.annotation.gtf.gz"

gtf = pd.read_csv(
    gtf_file,
    sep="\t",
    comment="#",
    header=None,
    names=["seqname", "source", "feature", "start", "end", "score", "strand", "frame", "attribute"])

  # 只保留 gene 层级
genes = gtf[gtf["feature"] == "gene"].copy()

  # 提取 gene_name 和 gene_id
genes["gene_symbol"] = genes["attribute"].str.extract(r'gene_name "([^"]+)"')
genes["gene_id"] = genes["attribute"].str.extract(r'gene_id "([^"]+)"')

  # 整理列
genes = genes[["gene_id", "gene_symbol", "seqname", "start", "end"]].rename(
    columns={"seqname": "chromosome"})

  # 去掉非标准染色体
genes = genes[genes["chromosome"].str.startswith("chr")].copy()

  # 保证每个 gene_symbol 只有一条记录
genes_unique = genes.drop_duplicates("gene_symbol", keep="first").set_index("gene_symbol")

  # 按 adata_epi.var.index（gene_symbol）对齐补充注释
adata_epi.var = adata_epi.var.join(genes_unique[["gene_id", "chromosome", "start", "end"]],how="left")

print(adata_epi.var.shape)
print(adata_epi.var[["gene_id", "chromosome", "start", "end"]].head())

In [ ]:
sc.pp.normalize_total(adata_epi, target_sum=1e4)
sc.pp.log1p(adata_epi)

In [ ]:
cnv.tl.infercnv(
    adata_epi,
    reference_key="status",
    reference_cat=["normal-like"],
    window_size=101,
    n_jobs=40,
)


In [ ]:
adata_epi.obs["cell_subtype_or_nl"] = np.where(
      adata_epi.obs["status"] == "normal-like",
      "normal-like",
      adata_epi.obs["cell_subtype"]
  )

In [ ]:
cnv.tl.pca(adata_epi,n_comps=50,use_rep='cnv',key_added='cnv_pca')
# ========= 构建邻居图 + UMAP =========
cnv.pp.neighbors(adata_epi, use_rep="cnv_pca", n_neighbors=15, key_added="cnv_neighbors",n_pcs=10)
cnv.tl.leiden(
      adata_epi,
      resolution=0.1,
      key_added="cnv_leiden",
      neighbors_key="cnv_neighbors",random_state=42,
      #flavor="igraph",
  )
cnv.tl.umap(adata_epi, neighbors_key="cnv_neighbors")
cnv.tl.cnv_score(adata_epi, groupby="cnv_leiden",key_added='cnv_score')

In [ ]:
# ========= GMM 自动分类 aneuploid / diploid =========
scores = adata_epi.obs["cnv_score"].values.reshape(-1, 1)
gmm = GaussianMixture(n_components=2, random_state=0).fit(scores)
labels = gmm.predict(scores)

means = gmm.means_.flatten()
aneuploid_label = np.argmax(means)
adata_epi.obs["cnv_state"] = np.where(labels == aneuploid_label, "Aneuploid", "Diploid")

print(adata_epi.obs["cnv_state"].value_counts())

In [ ]:
print(adata_epi.obs["cnv_leiden"].value_counts())

In [ ]:
cnv_summary = (
      adata_epi.obs.groupby("cell_subtype")["cnv_score"]
      .agg(["mean", "median", "count", "std"])
      .sort_values("mean")
  )
cnv_summary

In [ ]:
pd.crosstab(adata_epi.obs["status"], adata_epi.obs["cell_subtype"])

In [ ]:
adata_epi_t = adata_epi[adata_epi.obs["status"] == "tumor", :].copy()
adata_epi_t

cnv_summary_t = (
    adata_epi_t.obs.groupby("cell_subtype")["cnv_score"]
    .agg(["mean", "median", "count", "std"])
    .sort_values("mean")
)
print(cnv_summary_t)


In [ ]:
adata_epi_nl = adata_epi[adata_epi.obs["status"] == "normal-like", :].copy()
adata_epi_nl

cnv_summary_nl = (
    adata_epi_nl.obs.groupby("cell_subtype")["cnv_score"]
    .agg(["mean", "median", "count", "std"])
    .sort_values("mean")
)
print(cnv_summary_nl)


In [ ]:
adata_epi.write_h5ad('./adata_epi_cnv.h5ad')

# inferCNV selected visualization with editable text

Read `adata_epi_cnv.h5ad` and redraw the selected inferCNV heatmap plus 1 x 4 UMAP panels for all cells, tumor cells, and normal-like cells into `figures_text_redraw/`. PDF text uses TrueType fonts and SVG text remains editable.


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
sc.settings.set_figure_params(figsize=(3, 3), dpi=150, vector_friendly=True)

fig_dir = Path("figures_text_redraw")
fig_dir.mkdir(exist_ok=True)
sc.settings.figdir = str(fig_dir)

adata = sc.read_h5ad("./adata_epi_cnv.h5ad")
adata.obs["status"] = pd.Categorical(
    adata.obs["status"],
    categories=["tumor", "normal-like"],
    ordered=True,
)
if "cell_subtype_or_nl" not in adata.obs:
    adata.obs["cell_subtype_or_nl"] = np.where(
        adata.obs["status"] == "normal-like",
        "normal-like",
        adata.obs["cell_subtype"],
    )
adata.obs["cell_subtype_or_nl"] = adata.obs["cell_subtype_or_nl"].astype("category")
adata


In [ ]:
cnv.pl.chromosome_heatmap_summary(
    adata,
    groupby="cell_subtype_or_nl",
    vmin=-0.2,
    vmax=0.2,
    save="_epi_cellsubtype_nvst_sum.pdf",
)
cnv.pl.chromosome_heatmap_summary(
    adata,
    groupby="cell_subtype_or_nl",
    vmin=-0.2,
    vmax=0.2,
    save="_epi_cellsubtype_nvst_sum.svg",
)


In [ ]:
cnv_score_vmin = float(adata.obs["cnv_score"].min())
cnv_score_vmax = float(adata.obs["cnv_score"].max())

def save_umap_panel(subset, prefix):
    fig, axes = plt.subplots(1, 4, figsize=(12, 3), constrained_layout=True)
    umap_panels = [
        ("cell_subtype", "cell_subtype"),
        ("status", "status"),
        ("cnv_score", "cnv_score"),
        ("cnv_state", "cnv_state"),
    ]
    for ax, (key, title) in zip(axes, umap_panels):
        if key not in subset.obs:
            raise KeyError(f"Missing required obs column: {key}")
        plot_kwargs = {
            "vmin": cnv_score_vmin,
            "vmax": cnv_score_vmax,
        } if key == "cnv_score" else {}
        sc.pl.umap(
            subset,
            color=key,
            size=0.8,
            ax=ax,
            show=False,
            title=title,
            **plot_kwargs,
        )
        ax.set_aspect("equal", adjustable="box")
        ax.set_box_aspect(1)
    for suffix in ["pdf", "svg"]:
        out = fig_dir / f"{prefix}.{suffix}"
        fig.savefig(out, bbox_inches="tight")
    plt.close(fig)

save_umap_panel(adata, "umap_epi_all_cellsubtype_status_cnvscore_cnvstate")
save_umap_panel(adata[adata.obs["status"] == "tumor", :].copy(), "umap_epi_t_cellsubtype_status_cnvscore_cnvstate")
save_umap_panel(adata[adata.obs["status"] == "normal-like", :].copy(), "umap_epi_nl_cellsubtype_status_cnvscore_cnvstate")


### Final representative epithelial subtype dotplot

Final non-overlapping top3 marker dotplot based on epithelial subtype identity and checked against `cellsubtype_degs` DEG tables. This version excludes `Epi_CD44_high` and writes new output files without overwriting the original dotplot.


In [ ]:
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

base_dir = Path(".")
deg_dir = base_dir / "cellsubtype_degs"
fig_dir = base_dir / "figures"
fig_dir.mkdir(exist_ok=True)
sc.settings.figdir = str(fig_dir)

adata_representative = sc.read_h5ad(base_dir / "adata_epi.h5ad")
representative_order_no_cd44 = [
    "Epi_MIOX",
    "Epi_GPX3",
    "Epi_ALDOB",
    "Epi_CA9",
    "Epi_JUN",
    "Epi_VIM",
    "Epi_AQP2",
    "Epi_CA12",
]
representative_top3_markers_no_cd44 = {
    "Epi_MIOX": ["MIOX", "PCK1", "NAT8"],
    "Epi_GPX3": ["GPX3", "GATM", "ASS1"],
    "Epi_ALDOB": ["ALDOB", "FABP1", "APOE"],
    "Epi_CA9": ["CA9", "NDUFA4L2", "EGLN3"],
    "Epi_JUN": ["JUN", "FOS", "ATF3"],
    "Epi_VIM": ["VIM", "S100A10", "LGALS1"],
    "Epi_AQP2": ["AQP2", "FXYD4", "HSD11B2"],
    "Epi_CA12": ["CA12", "ATP6V0A4", "FOXI1"],
}
identity_rows = [
    ["Epi_MIOX", "Differentiated proximal tubule-like epithelium", "MIOX;PCK1;NAT8"],
    ["Epi_GPX3", "Proximal tubule antioxidant/metabolic epithelium", "GPX3;GATM;ASS1"],
    ["Epi_ALDOB", "Metabolic proximal tubule-like epithelium", "ALDOB;FABP1;APOE"],
    ["Epi_CA9", "Hypoxic ccRCC tumor-like epithelium", "CA9;NDUFA4L2;EGLN3"],
    ["Epi_JUN", "AP-1-high stress-response epithelial state", "JUN;FOS;ATF3"],
    ["Epi_VIM", "VIM-positive mesenchymal-like/invasive epithelial state", "VIM;S100A10;LGALS1"],
    ["Epi_AQP2", "Collecting duct principal cell-like epithelium", "AQP2;FXYD4;HSD11B2"],
    ["Epi_CA12", "Collecting duct intercalated cell-like epithelium", "CA12;ATP6V0A4;FOXI1"],
]
pd.DataFrame(
    identity_rows,
    columns=["subtype", "interpreted_identity", "selected_markers"],
).to_csv(fig_dir / "episubtype_identity_marker_summary_no_cd44.csv", index=False)

rows = []
all_genes = []
for subtype, genes in representative_top3_markers_no_cd44.items():
    deg = pd.read_csv(deg_dir / f"{subtype}_degs_epi.csv").set_index("gene")
    sub = adata_representative[adata_representative.obs["cell_subtype"] == subtype]
    X = sub.raw[:, genes].X
    own_mean = np.asarray(X.mean(axis=0)).ravel()
    own_pct = np.asarray((X > 0).mean(axis=0)).ravel()
    for gene, mean_value, pct_value in zip(genes, own_mean, own_pct):
        if gene not in deg.index:
            raise ValueError(f"{gene} not found in {subtype} DEG table")
        row = deg.loc[gene]
        if row["logfoldchanges"] <= 0 or row["pvals_adj"] >= 0.05:
            raise ValueError(f"{gene} is not a significant up-regulated DEG for {subtype}")
        rows.append({
            "subtype": subtype,
            "gene": gene,
            "own_mean": mean_value,
            "own_pct": pct_value,
            "score": row["score"],
            "logfoldchanges": row["logfoldchanges"],
            "pvals_adj": row["pvals_adj"],
            "selection_rule": "representative_interpretable_nonoverlapping_top3_checked_as_significant_up_DEG",
        })
        all_genes.append(gene)
if len(all_genes) != len(set(all_genes)):
    duplicated = pd.Series(all_genes)[pd.Series(all_genes).duplicated()].tolist()
    raise ValueError(f"Duplicated genes found: {duplicated}")

representative_marker_df = pd.DataFrame(rows)
representative_marker_df.to_csv(
    fig_dir / "episubtype_deg_markers_representative_top3_no_cd44.csv",
    index=False,
)

adata_representative_no_cd44 = adata_representative[
    adata_representative.obs["cell_subtype"] != "Epi_CD44_high"
].copy()
if hasattr(adata_representative_no_cd44.obs["cell_subtype"], "cat"):
    adata_representative_no_cd44.obs["cell_subtype"] = adata_representative_no_cd44.obs["cell_subtype"].cat.remove_unused_categories()

sc.pl.dotplot(
    adata_representative_no_cd44,
    var_names=representative_top3_markers_no_cd44,
    groupby="cell_subtype",
    standard_scale="var",
    use_raw=True,
    categories_order=representative_order_no_cd44,
    save="_episubtype_representative_top3_no_cd44.pdf",
)
representative_marker_df


### Epithelial and EMT-associated marker dotplot

Dotplot for epithelial identity, junction, and EMT-associated markers across epithelial subtypes. This uses `adata.raw` because several transcription-factor EMT markers are present in `.raw` but not in the filtered `.var`. `Epi_CD44_high` is excluded to match the representative epithelial subtype dotplot.


In [ ]:
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

base_dir = Path(".")
fig_dir = base_dir / "figures"
fig_dir.mkdir(exist_ok=True)
sc.settings.figdir = str(fig_dir)

adata_emt = sc.read_h5ad(base_dir / "adata_epi.h5ad")

# Ordered by the provided ascending values:
# Epi_MIOX 0.169859 < Epi_GPX3 0.217103 < Epi_ALDOB 0.249980
# < Epi_CA9 0.646181 < Epi_JUN 0.703860 < Epi_VIM 0.747663
emt_marker_order_no_cd44 = [
    "Epi_MIOX",
    "Epi_GPX3",
    "Epi_ALDOB",
    "Epi_CA9",
    "Epi_JUN",
    "Epi_VIM",
]
adata_emt = adata_emt[adata_emt.obs["cell_subtype"].isin(emt_marker_order_no_cd44)].copy()
if hasattr(adata_emt.obs["cell_subtype"], "cat"):
    adata_emt.obs["cell_subtype"] = adata_emt.obs["cell_subtype"].cat.remove_unused_categories()

epithelial_emt_markers = {
    "Epithelial / junction": ["CDH1", "OCLN", "CLDN1", "CLDN4", "EPCAM", "KRT8", "KRT18", "MUC1"],
    "EMT-associated": ["VIM", "CDH2", "SNAI1", "SNAI2", "TWIST1", "TWIST2", "ZEB1", "ZEB2", "FN1", "ACTA2", "MMP2", "MMP9"],
}

raw_genes = set(adata_emt.raw.var_names if adata_emt.raw is not None else [])
missing = [gene for genes in epithelial_emt_markers.values() for gene in genes if gene not in raw_genes]
if missing:
    raise ValueError(f"Markers missing from adata.raw: {missing}")

pd.DataFrame(
    [
        {"marker_group": group, "gene": gene}
        for group, genes in epithelial_emt_markers.items()
        for gene in genes
    ]
).to_csv(fig_dir / "episubtype_epithelial_emt_markers.csv", index=False)

sc.pl.dotplot(
    adata_emt,
    var_names=epithelial_emt_markers,
    groupby="cell_subtype",
    standard_scale="var",
    use_raw=True,
    dot_max=1,
    categories_order=emt_marker_order_no_cd44,
    save="_episubtype_epithelial_emt_markers_no_cd44.pdf",
)
